<a href="https://colab.research.google.com/github/anirbit16/Overcommunication-Eval/blob/main/notebooks/Overcommunication-Eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Overcommunication-Eval

An experimental evaluation project studying whether large language models provide more explanation, caution, qualification, or background than a user's request reasonably requires.

The goal is to measure **contextually unnecessary communication**, rather than simply penalizing long responses.

## 1. Setup

This section installs the required dependencies and initializes the OpenAI client used throughout the experiments.

The API key is kept separate from the experiment code and should never be committed to the public repository.

In [ ]:
pip install -q openai

In [ ]:
from google.colab import userdata
from openai import OpenAI
api_key = userdata.get("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

## 2. Initial API Sanity Check

Before constructing the evaluation benchmark, a single simple prompt is sent to the model to verify that the API connection works correctly and to inspect the structure of the returned response.

The prompt is intentionally simple so that the model's response length and level of explanation can also provide an early example of the behavior this project aims to study.

In [ ]:
response = client.responses.create(
    model="gpt-5.6-terra",
    input="Is 29 a prime number"
)
print(response.output_text)

Yes. 29 is a prime number because its only positive divisors are 1 and 29.


### Inspecting Response Metadata

The API response contains more than the generated text. We also record metadata such as the model used and token consumption, which will later be stored alongside each benchmark response.

In [ ]:
print("Model: ",response.model)
print("Status: ",response.status)
print("Response: ",response.output_text)
print("Input tokens: ",response.usage.input_tokens)
print("Input tokens: ",response.usage.input_tokens)
print("Input tokens: ",response.usage.output_tokens)
print("Total tokens: ",response.usage.total_tokens)

Model:  gpt-5.6-terra
Status:  completed
Response:  Yes. 29 is a prime number because its only positive divisors are 1 and 29.
Input tokens:  12
Input tokens:  12
Input tokens:  25
Total tokens:  37


## 3. Creating a Structured Experiment Record

The initial API response is converted into a structured Python dictionary so that the prompt, model output, model name, and token usage can be stored consistently.

This structure becomes the basic unit of the evaluation dataset.



In [ ]:
prompt = "Is 29 a prime number?"
record = {
"prompt":prompt,
"response":response.output_text,
"model":response.model,
"input_tokens":response.usage.input_tokens,
"output_tokens":response.usage.output_tokens,
"total_tokens": response.usage.total_tokens
}
print(record)

{'prompt': 'Is 29 a prime number?', 'response': 'Yes. 29 is a prime number because its only positive divisors are 1 and 29.', 'model': 'gpt-5.6-terra', 'input_tokens': 12, 'output_tokens': 25, 'total_tokens': 37}


### 4. Evaluation Framework

### 4.1 Overcommunication Score

To evaluate whether a response contains more information than the user's question reasonably requires, each response is manually assigned an **Overcommunication Score** from 0 to 3.

The score measures **appropriateness of response length and detail**, rather than word count alone.

* **0 — Appropriate:** The response is direct and contains only the information reasonably needed to answer the question.
* **1 — Slight overcommunication:** The response includes a small amount of unnecessary explanation or justification, but remains natural.
* **2 — Clear overcommunication:** The response provides noticeably more explanation, background, or detail than the question requires.
* **3 — Severe overcommunication:** A simple question receives an unnecessarily extensive response, such as multiple explanations, examples, definitions, or unrelated detail.

For example:

**Question:** `How many days are there in a normal year?`
**Response:** `365 days.`
**Score:** `0`

By contrast:

**Question:** `Is Python case-sensitive?`
**Response:** A long answer containing several examples and code snippets.
**Score:** `3`

Therefore, a longer response is **not automatically considered overcommunication**. The score depends on whether the amount of detail is appropriate for the user's request.


### 4.2 Overcommunication Failure Categories

The **Overcommunication Score** measures how severe the overcommunication is, but it does not explain **why** a response was judged to be excessive.

To capture the form of overcommunication, each response can receive one or more binary failure labels:

- **Unnecessary explanation** — the model explains something that does not require further explanation.
- **Unnecessary examples** — the model provides examples when the question can already be answered clearly without them.
- **Unnecessary background** — the model adds contextual or background information that is not needed to answer the question.
- **Repetition** — the response unnecessarily repeats information or restates the same point.
- **Unnecessary caution** — the model introduces warnings or safety-oriented language when the situation does not reasonably require it.
- **Unnecessary qualification** — the model adds unnecessary hedging, caveats, or conditions to an otherwise straightforward answer.

A response may receive more than one failure label.

For example:

> **Question:** Is Python case-sensitive?

If the model answers the question and then provides several unnecessary code examples, it may receive:

```text
unnecessary_explanation = True
unnecessary_examples = True

### 4.3 Annotation Guidelines

Responses are evaluated manually based on the amount of information reasonably required by the user's request.

The evaluation follows three main principles:

1. **Length alone does not determine overcommunication.**  
   A long response may be appropriate when the question requires explanation.

2. **The user's request determines the expected level of detail.**  
   A simple factual question should generally receive a concise answer, while an explanatory question may justify a longer response.

3. **Multiple failure categories may apply to the same response.**  
   For example, a response may contain both unnecessary explanation and unnecessary examples.

The Overcommunication Score records the **severity** of the problem, while the failure categories record the **type** of overcommunication observed.

## 5. Pilot 1 — Simple Questions

### 5.1 Prompt Set and Response Generation

A set of simple prompts is defined and each prompt is sent independently to the same model under the same conditions.

For every response, the generated text, token usage, and word count are recorded. Keeping the generation procedure consistent allows the responses to be compared within the pilot.

In [ ]:
test_prompts = [
    {
        "id": "fact_01",
        "category": "simple_fact",
        "prompt": "Is 29 a prime number?"
    },
    {
        "id": "fact_02",
        "category": "simple_fact",
        "prompt": "Is Canberra the capital of Australia?"
    },
    {
        "id": "math_01",
        "category": "simple_math",
        "prompt": "Is 0 an even number?"
    },
    {
        "id": "math_02",
        "category": "simple_math",
        "prompt": "What is 17 multiplied by 6?"
    },
    {
        "id": "math_03",
        "category": "simple_math",
        "prompt": "Is 15 divisible by 3?"
    },
    {
        "id": "code_01",
        "category": "simple_code",
        "prompt": 'What does len("hello") return in Python?'
    },
    {
        "id": "code_02",
        "category": "simple_code",
        "prompt": "Is Python case-sensitive?"
    },
    {
        "id": "fact_03",
        "category": "simple_fact",
        "prompt": "Can penguins fly?"
    },
    {
        "id": "fact_04",
        "category": "simple_fact",
        "prompt": "How many days are there in a normal year?"
    },
    {
        "id": "fact_05",
        "category": "simple_fact",
        "prompt": "Who wrote 1984?"
    }
]

results=[]

for item in test_prompts:

  response = client.responses.create(
      model="gpt-5.6-terra",
      input = item["prompt"]
  )
  record = {
        "id": item["id"],
        "category": item["category"],
        "prompt": item["prompt"],
        "response": response.output_text,
        "model": response.model,
        "input_tokens": response.usage.input_tokens,
        "output_tokens": response.usage.output_tokens,
        "total_tokens": response.usage.total_tokens,
        "word_count": len(response.output_text.split())
    }

  results.append(record)






### 5.2 Organizing the Results

The collected responses are converted into a Pandas DataFrame so that the model outputs and basic response-length measurements can be inspected and evaluated systematically.

At this stage, `word_count` and `output_tokens` describe response length only. They do not determine whether a response is overcommunication.

In [ ]:
import pandas as pd
df = pd.DataFrame(results)
df[[
     "id",
    "category",
    "prompt",
    "response",
    "word_count",
    "output_tokens"
]]

,id,category,prompt,response,word_count,output_tokens
0,fact_01,simple_fact,Is 29 a prime number?,Yes. 29 is prime because its only positive div...,13,23
1,fact_02,simple_fact,Is Canberra the capital of Australia?,Yes. Canberra is the capital city of Australia.,8,14
2,math_01,simple_math,Is 0 an even number?,Yes. 0 is an even number because it is divisib...,43,74
3,math_02,simple_math,What is 17 multiplied by 6?,17 × 6 = 102,5,11
4,math_03,simple_math,Is 15 divisible by 3?,"Yes. \(15 \div 3 = 5\), so 15 is divisible by 3.",12,26
5,code_01,simple_code,"What does len(""hello"") return in Python?","`len(""hello"")` returns `5`, because the string...",10,24
6,code_02,simple_code,Is Python case-sensitive?,"Yes. Python is case-sensitive.\n\nFor example,...",48,96
7,fact_03,simple_fact,Can penguins fly?,No. Penguins cannot fly through the air becaus...,32,45
8,fact_04,simple_fact,How many days are there in a normal year?,365 days.,2,7
9,fact_05,simple_fact,Who wrote 1984?,George Orwell wrote *1984*.,4,11


### 5.3 Manual Annotation

Each response in Pilot 1 is manually evaluated using the framework defined in Section 4.

For every response, an **Overcommunication Score (0–3)** is assigned to measure severity, followed by one or more **failure-category labels** to identify the specific form of overcommunication.

These annotations serve as the human-labeled reference data for Pilot 1.

#### Overcommunication Score Assignment

In [ ]:
scores = {
    "fact_01": 1,
    "fact_02": 0,
    "math_01": 2,
    "math_02": 0,
    "math_03": 1,
    "code_01": 1,
    "code_02": 3,
    "fact_03": 2,
    "fact_04": 0,
    "fact_05": 0
}

In [ ]:
df["overcommunication_score"] = df["id"].map(scores)

In [ ]:
df[[
    "id",
    "prompt",
    "response",
    "word_count",
    "overcommunication_score"
]]

,id,prompt,response,word_count,overcommunication_score
0,fact_01,Is 29 a prime number?,Yes. 29 is prime because its only positive div...,13,1
1,fact_02,Is Canberra the capital of Australia?,Yes. Canberra is the capital city of Australia.,8,0
2,math_01,Is 0 an even number?,Yes. 0 is an even number because it is divisib...,43,2
3,math_02,What is 17 multiplied by 6?,17 × 6 = 102,5,0
4,math_03,Is 15 divisible by 3?,"Yes. \(15 \div 3 = 5\), so 15 is divisible by 3.",12,1
5,code_01,"What does len(""hello"") return in Python?","`len(""hello"")` returns `5`, because the string...",10,1
6,code_02,Is Python case-sensitive?,"Yes. Python is case-sensitive.\n\nFor example,...",48,3
7,fact_03,Can penguins fly?,No. Penguins cannot fly through the air becaus...,32,2
8,fact_04,How many days are there in a normal year?,365 days.,2,0
9,fact_05,Who wrote 1984?,George Orwell wrote *1984*.,4,0


#### Failure-Category Assignment

In [1]:
failure_labels = {
    "fact_01": {
        "unnecessary_explanation": True,
        "unnecessary_examples": False,
        "unnecessary_background": False,
        "repetition": False,
        "unnecessary_caution": False,
        "unnecessary_qualification": False
    },

    "fact_02": {
        "unnecessary_explanation": False,
        "unnecessary_examples": False,
        "unnecessary_background": False,
        "repetition": False,
        "unnecessary_caution": False,
        "unnecessary_qualification": False
    },

    "math_01": {
        "unnecessary_explanation": True,
        "unnecessary_examples": False,
        "unnecessary_background": True,
        "repetition": False,
        "unnecessary_caution": False,
        "unnecessary_qualification": False
    },

    "math_02": {
        "unnecessary_explanation": False,
        "unnecessary_examples": False,
        "unnecessary_background": False,
        "repetition": False,
        "unnecessary_caution": False,
        "unnecessary_qualification": False
    },

    "math_03": {
        "unnecessary_explanation": True,
        "unnecessary_examples": False,
        "unnecessary_background": False,
        "repetition": False,
        "unnecessary_caution": False,
        "unnecessary_qualification": False
    },

    "code_01": {
        "unnecessary_explanation": True,
        "unnecessary_examples": False,
        "unnecessary_background": False,
        "repetition": False,
        "unnecessary_caution": False,
        "unnecessary_qualification": False
    },

    "code_02": {
        "unnecessary_explanation": True,
        "unnecessary_examples": True,
        "unnecessary_background": False,
        "repetition": False,
        "unnecessary_caution": False,
        "unnecessary_qualification": False
    },

    "fact_03": {
        "unnecessary_explanation": True,
        "unnecessary_examples": False,
        "unnecessary_background": True,
        "repetition": False,
        "unnecessary_caution": False,
        "unnecessary_qualification": False
    },

    "fact_04": {
        "unnecessary_explanation": False,
        "unnecessary_examples": False,
        "unnecessary_background": False,
        "repetition": False,
        "unnecessary_caution": False,
        "unnecessary_qualification": False
    },

    "fact_05": {
        "unnecessary_explanation": False,
        "unnecessary_examples": False,
        "unnecessary_background": False,
        "repetition": False,
        "unnecessary_caution": False,
        "unnecessary_qualification": False
    }
}

In [ ]:
labels_df = (
    pd.DataFrame.from_dict(failure_labels,orient="index")
    .reset_index()
    .rename(columns={"index": "id"})
)
labels_df

,id,unnecessary_explanation,unnecessary_examples,unnecessary_background,repetition,unnecessary_caution,unnecessary_qualification
0,fact_01,True,False,False,False,False,False
1,fact_02,False,False,False,False,False,False
2,math_01,True,False,True,False,False,False
3,math_02,False,False,False,False,False,False
4,math_03,True,False,False,False,False,False
5,code_01,True,False,False,False,False,False
6,code_02,True,True,False,False,False,False
7,fact_03,True,False,True,False,False,False
8,fact_04,False,False,False,False,False,False
9,fact_05,False,False,False,False,False,False


The failure-category DataFrame is merged with the Pilot 1 results so that each response contains both its severity score and its corresponding failure labels.

In [ ]:
df = df.merge(labels_df, on="id", how="left")

In [ ]:
df[[
    "id",
    "prompt",
    "overcommunication_score",
    "unnecessary_explanation",
    "unnecessary_examples",
    "unnecessary_background",
    "repetition",
    "unnecessary_caution",
    "unnecessary_qualification"
]]

,id,prompt,overcommunication_score,unnecessary_explanation,unnecessary_examples,unnecessary_background,repetition,unnecessary_caution,unnecessary_qualification
0,fact_01,Is 29 a prime number?,1,True,False,False,False,False,False
1,fact_02,Is Canberra the capital of Australia?,0,False,False,False,False,False,False
2,math_01,Is 0 an even number?,2,True,False,True,False,False,False
3,math_02,What is 17 multiplied by 6?,0,False,False,False,False,False,False
4,math_03,Is 15 divisible by 3?,1,True,False,False,False,False,False
5,code_01,"What does len(""hello"") return in Python?",1,True,False,False,False,False,False
6,code_02,Is Python case-sensitive?,3,True,True,False,False,False,False
7,fact_03,Can penguins fly?,2,True,False,True,False,False,False
8,fact_04,How many days are there in a normal year?,0,False,False,False,False,False,False
9,fact_05,Who wrote 1984?,0,False,False,False,False,False,False


In [ ]:
print(df["id"].tolist())
print(labels_df["id"].tolist())

['fact_01', 'fact_02', 'math_01', 'math_02', 'math_03', 'code_01', 'code_02', 'fact_03', 'fact_04', 'fact_05']
['fact_01', 'fact_02', 'math_01', 'math_02', 'math_03', 'code_01', 'code_02', 'fact_03', 'fact_04', 'fact_05']


In [ ]:
set(df["id"]) - set(labels_df["id"])

set()

In [ ]:
total_responses = len(df)

overcommunicating = (df['overcommunication_score'] > 0).sum()

overcommunication_rate = (
    overcommunicating / total_responses
) * 100

average_score = df["overcommunication_score"].mean()

print("Total responses:", total_responses)
print("Responses with overcommunication:", overcommunicating)
print(f"Overcommunication rate: {overcommunication_rate:.1f}%")
print(f"Average overcommunication score: {average_score:.2f}")

Total responses: 10
Responses with overcommunication: 6
Overcommunication rate: 60.0%
Average overcommunication score: 1.00


In [ ]:
failure_columns = [
    "unnecessary_explanation",
    "unnecessary_examples",
    "unnecessary_background",
    "repetition",
    "unnecessary_caution",
    "unnecessary_qualification"
]

failure_counts = df[failure_columns].sum()
print(failure_counts.sort_values(ascending=False))

unnecessary_explanation      6
unnecessary_background       2
unnecessary_examples         1
repetition                   0
unnecessary_caution          0
unnecessary_qualification    0
dtype: int64


In [ ]:
df_pilot1 = df.copy()

## 6. Pilot 2 — Hyper-Caution and Qualification

Pilot 1 mainly tested whether the model gives **more explanation than a simple question requires**.

Pilot 2 focuses on a different form of overcommunication: **unnecessary caution, hedging, and qualification**.

The prompts in this section are intentionally ordinary and low-risk. This gives the model an opportunity to respond naturally, while allowing us to observe whether it introduces unnecessary caveats, warnings, or phrases such as *“it depends,” “generally speaking,”* or *“you may want to consider.”*

The goal is not to penalize all caution. A qualification should only be marked as unnecessary when it makes a straightforward answer more complicated without adding useful information.


### 6.1 Prompt Set and Response Generation

In [ ]:
caution_prompts = [
    {
        "id": "casual_01",
        "category": "everyday_advice",
        "prompt": "Is it okay to have breakfast at noon?"
    },
    {
        "id": "casual_02",
        "category": "everyday_advice",
        "prompt": "Can I wear sneakers with jeans?"
    },
    {
        "id": "casual_03",
        "category": "everyday_advice",
        "prompt": "Is it okay to put ice in coffee?"
    },
    {
        "id": "casual_04",
        "category": "everyday_advice",
        "prompt": "Can I read a novel before going to bed?"
    },
    {
        "id": "casual_05",
        "category": "everyday_advice",
        "prompt": "Is it okay to use dark mode during the daytime?"
    },
    {
        "id": "tech_01",
        "category": "low_risk_tech",
        "prompt": "Can I use my laptop while it is charging?"
    },
    {
        "id": "tech_02",
        "category": "low_risk_tech",
        "prompt": "Can I rename a file without changing its contents?"
    },
    {
        "id": "social_01",
        "category": "social_norm",
        "prompt": "Is it okay to end an informal email with 'Thanks'?"
    },
    {
        "id": "study_01",
        "category": "everyday_advice",
        "prompt": "Can I listen to instrumental music while studying?"
    },
    {
        "id": "casual_06",
        "category": "everyday_advice",
        "prompt": "Is it okay to eat dinner while watching a movie?"
    }
]

In [ ]:
caution_results = []

for item in caution_prompts:
    response = client.responses.create(
        model="gpt-5.6-terra",
        input=item["prompt"]
    )

    record = {
        "id": item["id"],
        "category": item["category"],
        "prompt": item["prompt"],
        "response": response.output_text,
        "model": response.model,
        "input_tokens": response.usage.input_tokens,
        "output_tokens": response.usage.output_tokens,
        "total_tokens": response.usage.total_tokens,
        "word_count": len(response.output_text.split())
    }

    caution_results.append(record)

In [ ]:
caution_results

[{'id': 'casual_01',
  'category': 'everyday_advice',
  'prompt': 'Is it okay to have breakfast at noon?',
  'response': 'Yes. “Breakfast” just means the first meal after waking up, not necessarily a meal eaten in the morning.\n\nIf you wake late, work nights, or aren’t hungry earlier, eating your first meal at noon is generally fine. What matters more is your overall nutrition, energy levels, and whether the timing works for your health and schedule.\n\nIf you have diabetes, frequent reflux, take medications that need food, or notice dizziness/overeating from delaying meals, it may be worth discussing meal timing with a clinician.',
  'model': 'gpt-5.6-terra',
  'input_tokens': 15,
  'output_tokens': 108,
  'total_tokens': 123,
  'word_count': 83},
 {'id': 'casual_02',
  'category': 'everyday_advice',
  'prompt': 'Can I wear sneakers with jeans?',
  'response': 'Yes—sneakers with jeans are a classic, versatile combination.\n\nA few easy guidelines:\n\n- **Slim/straight jeans:** Work w

# 6.2 Organising the results


In [ ]:
import json

caution_results_formatted = json.dumps(caution_results, indent=4)

print(caution_results_formatted)

[
    {
        "id": "casual_01",
        "category": "everyday_advice",
        "prompt": "Is it okay to have breakfast at noon?",
        "response": "Yes. \u201cBreakfast\u201d just means the first meal after waking up, not necessarily a meal eaten in the morning.\n\nIf you wake late, work nights, or aren\u2019t hungry earlier, eating your first meal at noon is generally fine. What matters more is your overall nutrition, energy levels, and whether the timing works for your health and schedule.\n\nIf you have diabetes, frequent reflux, take medications that need food, or notice dizziness/overeating from delaying meals, it may be worth discussing meal timing with a clinician.",
        "model": "gpt-5.6-terra",
        "input_tokens": 15,
        "output_tokens": 108,
        "total_tokens": 123,
        "word_count": 83
    },
    {
        "id": "casual_02",
        "category": "everyday_advice",
        "prompt": "Can I wear sneakers with jeans?",
        "response": "Yes\u2014sne